In [1]:
import scanpy as sc

/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWa

In [2]:
def add_subgroup_annotations(adata_train, adata): 

    train_conditions = adata_train.obs.condition.str.replace("+ctrl", "").str.replace("ctrl+", "").unique()

    assert not adata[adata.obs.condition != "ctrl"].obs.condition.isin(train_conditions).any()

    mask_single_perturbation = adata.obs.condition.str.contains("ctrl")
    mask_double_perturbation_seen_0 = (
        ~adata.obs.condition.str.contains("ctrl") & 
        ~adata.obs.gene_1.isin(train_conditions) & 
        ~adata.obs.gene_2.isin(train_conditions)
    )
    mask_double_perturbation_seen_1 = (
        ~adata.obs.condition.str.contains("ctrl") & 
        (
            (adata.obs.gene_1.isin(train_conditions) & ~adata.obs.gene_2.isin(train_conditions)) | 
            (~adata.obs.gene_1.isin(train_conditions) & adata.obs.gene_2.isin(train_conditions))
        )
    )
    mask_double_perturbation_seen_2 = (
        ~adata.obs.condition.str.contains("ctrl") & 
        adata.obs.gene_1.isin(train_conditions) & 
        adata.obs.gene_2.isin(train_conditions)
    )
    adata.obs.loc[mask_single_perturbation, "subgroup"] = "single"
    adata.obs.loc[mask_double_perturbation_seen_0, "subgroup"] = "double_seen_0"
    adata.obs.loc[mask_double_perturbation_seen_1, "subgroup"] = "double_seen_1"
    adata.obs.loc[mask_double_perturbation_seen_2, "subgroup"] = "double_seen_2"


In [3]:
for split in range(5):
    adata_train_path = f"/home/haicu/soeren.becker/repos/ot_pert_reproducibility/norman2019/norman_preprocessed_adata/adata_train_pca_50_split_{split}.h5ad"
    adata_test_path = f"/home/haicu/soeren.becker/repos/ot_pert_reproducibility/norman2019/norman_preprocessed_adata/adata_val_pca_50_split_{split}.h5ad"
    adata_ood_path = f"/home/haicu/soeren.becker/repos/ot_pert_reproducibility/norman2019/norman_preprocessed_adata/adata_test_pca_50_split_{split}.h5ad"
    
    adata_train_path_new = f"/lustre/groups/ml01/workspace/ot_perturbation/data/norman_new/adata_train_pca_50_split_{split}.h5ad"
    adata_test_path_new = f"/lustre/groups/ml01/workspace/ot_perturbation/data/norman_new/adata_test_pca_50_split_{split}.h5ad"
    adata_ood_path_new = f"/lustre/groups/ml01/workspace/ot_perturbation/data/norman_new/adata_ood_pca_50_split_{split}.h5ad"
    
    adata_train = sc.read_h5ad(adata_train_path)
    adata_test = sc.read_h5ad(adata_test_path)
    adata_ood = sc.read_h5ad(adata_ood_path)

    add_subgroup_annotations(adata_train, adata_ood)

    adata_train.write_h5ad(adata_train_path_new)
    adata_test.write_h5ad(adata_test_path_new)
    adata_ood.write_h5ad(adata_ood_path_new)
